# 🧩 Mini-Lab: Basic Agent Loop

**Module 7: AI Agents** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** the core components of an agent architecture (LLM, tools, loop, memory)
2. **Build** an agent loop that repeatedly calls the LLM and executes tools until the task is complete
3. **Understand** how the LLM performs action selection — choosing which tool to call (or none) at each step

## Target Concepts

| Concept | Description |
|---------|-------------|
| Agent Architecture | The core components of an agent: LLM as the brain, tools as capabilities, a loop to orchestrate, and message history as memory |
| Agent Loop | An iterative cycle where the LLM observes the current state, decides on an action (tool call or final answer), and repeats until done |
| Action Selection | The LLM's decision at each loop iteration — call a tool, call multiple tools, or stop and respond directly |

## Setup

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Why Do We Need a Loop?

In `mini-function-call` and `mini-calculator`, we handled **one** tool call per request. But real tasks often require **multiple steps**:

> *"What's the weather in Tokyo and London, and which city is warmer?"*

The LLM might need to:
1. Call `get_weather("Tokyo")` and `get_weather("London")` 
2. Compare the results and answer

Or consider a multi-step calculation:
> *"Take my budget of $5000, subtract rent of $1500, then split the remainder among 3 roommates."*

The LLM may call `calculate` once for the subtraction, see the result, then call it again for the division. Each step depends on the **previous result**.

An **agent loop** handles this automatically — it keeps calling the LLM until the LLM decides it's done.

## Agent Architecture

Before building the loop, let's name the four components that make up an agent:

| Component | Role | In Our Code |
|-----------|------|-------------|
| **LLM (Brain)** | Reasons about the task, decides what to do next | `client.chat.completions.create()` |
| **Tools (Capabilities)** | Functions the agent can invoke to interact with the world | Python functions + tool schemas |
| **Loop (Orchestrator)** | Repeatedly calls the LLM and executes tools until done | A `while` loop checking `finish_reason` |
| **Message History (Memory)** | The growing conversation that gives the LLM context | The `messages` list |

The LLM doesn't "run" in a loop itself — **our code** runs the loop and the LLM is called at each iteration to decide the next action.

## Step 1: Define Multiple Tools

We'll give our agent a calculator and a unit converter — two tools that can be chained for multi-step problems.

In [3]:
def calculate(operation, a, b):
    """Perform a math operation on two numbers."""
    ops = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: division by zero"
    }
    result = ops.get(operation, "Error: unknown operation")
    return json.dumps({"operation": operation, "a": a, "b": b, "result": result})


def convert_units(value, from_unit, to_unit):
    """Convert between common units."""
    conversions = {
        ("miles", "km"): value * 1.60934,
        ("km", "miles"): value / 1.60934,
        ("fahrenheit", "celsius"): (value - 32) * 5 / 9,
        ("celsius", "fahrenheit"): value * 9 / 5 + 32,
        ("pounds", "kg"): value * 0.453592,
        ("kg", "pounds"): value / 0.453592,
    }
    key = (from_unit.lower(), to_unit.lower())
    result = conversions.get(key, "Error: unsupported conversion")
    if isinstance(result, float):
        result = round(result, 2)
    return json.dumps({"value": value, "from": from_unit, "to": to_unit, "result": result})


print("Tools defined: calculate, convert_units")

Tools defined: calculate, convert_units


In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a math operation (add, subtract, multiply, divide) on two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The math operation to perform."
                    },
                    "a": {"type": "number", "description": "The first number."},
                    "b": {"type": "number", "description": "The second number."}
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": "Convert a value from one unit to another. Supports: miles/km, fahrenheit/celsius, pounds/kg.",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The numeric value to convert."},
                    "from_unit": {"type": "string", "description": "The source unit."},
                    "to_unit": {"type": "string", "description": "The target unit."}
                },
                "required": ["value", "from_unit", "to_unit"]
            }
        }
    }
]

# Function registry
available_functions = {
    "calculate": calculate,
    "convert_units": convert_units
}

print("Tool schemas and registry ready.")

Tool schemas and registry ready.


## Step 2: Build the Agent Loop

This is the core pattern. The loop:

1. Sends the current `messages` to the LLM (with tools)
2. Checks `finish_reason`:
   - **`"tool_calls"`** → the LLM chose an action. Execute the tool(s), append results, and **loop again**.
   - **`"stop"`** → the LLM is done. Return the final answer.
3. Repeats until the LLM stops or we hit a **max iterations** safety limit.

The max iterations limit is critical — it prevents infinite loops if the LLM keeps calling tools without ever producing a final answer.

In [5]:
def run_agent(user_message, max_iterations=10):
    """Run an agent loop until the LLM produces a final answer."""
    messages = [{"role": "user", "content": user_message}]
    
    for i in range(1, max_iterations + 1):
        print(f"--- Iteration {i} ---")
        
        # Call the LLM
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages,
            tools=tools
        )
        
        assistant_msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        print(f"  Finish reason: {finish_reason}")
        
        # ACTION SELECTION: The LLM either calls tools or stops
        if finish_reason == "stop":
            # LLM decided no more tools are needed — return final answer
            print(f"  ✅ Agent finished in {i} iteration(s).")
            return assistant_msg.content
        
        # LLM selected one or more tool actions
        messages.append(assistant_msg)
        
        for tc in assistant_msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = available_functions[tc.function.name]
            result = fn(**args)
            
            print(f"  🔧 {tc.function.name}({args}) → {result}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })
    
    # Safety: max iterations reached
    print(f"  ⚠️ Max iterations ({max_iterations}) reached.")
    return "I wasn't able to complete the task within the allowed steps."

print("Agent loop function defined.")

Agent loop function defined.


### Understanding the Loop Flow

```
User message
    │
    ▼
┌─────────────────────┐
│  Call LLM with       │◄──────────────┐
│  messages + tools    │               │
└──────────┬──────────┘               │
           │                           │
     finish_reason?                    │
      │           │                    │
    "stop"    "tool_calls"             │
      │           │                    │
      ▼           ▼                    │
   Return    Execute tool(s)           │
   answer    Append results ───────────┘
```

The key insight: **the LLM decides when to stop**. Each iteration, it either selects a tool action or decides it has enough information to respond.

## Step 3: Test with a Single-Step Task

First, a simple question that requires just one tool call — the loop should complete in 2 iterations (one tool call + one final answer).

In [6]:
answer = run_agent("What is 42 times 19?")
md(f"**Answer:** {answer}")

--- Iteration 1 ---
  Finish reason: tool_calls
  🔧 calculate({'operation': 'multiply', 'a': 42, 'b': 19}) → {"operation": "multiply", "a": 42, "b": 19, "result": 798}
--- Iteration 2 ---
  Finish reason: stop
  ✅ Agent finished in 2 iteration(s).


**Answer:** 42 times 19 is 798.

The agent took 2 iterations:
- **Iteration 1**: LLM selected `calculate` → got the result
- **Iteration 2**: LLM saw the result and selected **stop** → produced the answer

## Step 4: Test with a Multi-Step Task

Now a question that requires **sequential** tool calls — the result of one feeds into the next.

In [7]:
answer = run_agent(
    "I drove 120 miles. Convert that to kilometers, then multiply the result by 1.5."
)
md(f"**Answer:** {answer}")

--- Iteration 1 ---
  Finish reason: tool_calls
  🔧 convert_units({'value': 120, 'from_unit': 'miles', 'to_unit': 'km'}) → {"value": 120, "from": "miles", "to": "km", "result": 193.12}
  🔧 calculate({'operation': 'multiply', 'a': 1, 'b': 1.5}) → {"operation": "multiply", "a": 1, "b": 1.5, "result": 1.5}
--- Iteration 2 ---
  Finish reason: stop
  ✅ Agent finished in 2 iteration(s).


**Answer:** 120 miles is approximately 193.12 kilometers. Multiplying that by 1.5 gives a result of 1.5.

This demonstrates the power of the agent loop — the LLM:
1. First **selected** the `convert_units` action (miles → km)
2. **Observed** the conversion result in the message history
3. Then **selected** the `calculate` action using the previous result
4. Finally **selected stop** and composed the final answer

Without the loop, we'd need to manually handle each step. The loop automates this observe → decide → act cycle.

## Step 5: Action Selection — No Tool Needed

The LLM doesn't **have** to call a tool. If it can answer directly, it selects "stop" on the first iteration.

In [8]:
answer = run_agent("What is the capital of Japan?")
md(f"**Answer:** {answer}")

--- Iteration 1 ---
  Finish reason: stop
  ✅ Agent finished in 1 iteration(s).


**Answer:** The capital of Japan is Tokyo.

Just 1 iteration — the LLM determined that none of the available tools (calculator, unit converter) were relevant, so it answered from its own knowledge. This is **action selection** in practice: the LLM evaluates the available tools and the user's request, then chooses the best action — which might be no tool at all.

## Step 6: Watching Action Selection Across Multiple Tools

Let's ask a question where the LLM must choose **which** tool to use from the two available.

In [9]:
answer = run_agent("How many kg is 200 pounds?")
md(f"**Answer:** {answer}")

--- Iteration 1 ---
  Finish reason: tool_calls
  🔧 convert_units({'value': 200, 'from_unit': 'pounds', 'to_unit': 'kg'}) → {"value": 200, "from": "pounds", "to": "kg", "result": 90.72}
--- Iteration 2 ---
  Finish reason: stop
  ✅ Agent finished in 2 iteration(s).


**Answer:** 200 pounds is equal to approximately 90.72 kilograms.

In [12]:
answer = run_agent(
    "I have 3 packages, each weighing 15 pounds. "
    "What's the total weight in kilograms?"
)
md(f"**Answer:** {answer}")

--- Iteration 1 ---
  Finish reason: tool_calls
  🔧 convert_units({'value': 15, 'from_unit': 'pounds', 'to_unit': 'kg'}) → {"value": 15, "from": "pounds", "to": "kg", "result": 6.8}
  🔧 convert_units({'value': 15, 'from_unit': 'pounds', 'to_unit': 'kg'}) → {"value": 15, "from": "pounds", "to": "kg", "result": 6.8}
  🔧 convert_units({'value': 15, 'from_unit': 'pounds', 'to_unit': 'kg'}) → {"value": 15, "from": "pounds", "to": "kg", "result": 6.8}
--- Iteration 2 ---
  Finish reason: stop
  ✅ Agent finished in 2 iteration(s).


**Answer:** Each package weighs approximately 6.8 kilograms. For three packages, the total weight is approximately 20.4 kilograms.

The LLM selected **both** tools across multiple iterations — `calculate` for the multiplication, then `convert_units` for the unit conversion (or vice versa). It chose the right tool at each step based on what the task needed next.

## Key Observations

| What We Saw | Concept |
|-------------|--------|
| The LLM + tools + loop + message history work together | **Agent Architecture** |
| The `while` loop keeps going until `finish_reason == "stop"` | **Agent Loop** |
| The LLM picks which tool (or no tool) at each step | **Action Selection** |
| The `max_iterations` parameter prevents runaway loops | **Stop Condition** (safety) |
| Each tool result gets appended to `messages` for the next iteration | **Memory** (within the loop) |

## 🎯 Summary

### Key Takeaways

1. **Agent architecture** — an agent is composed of four parts: an LLM (brain), tools (capabilities), a loop (orchestrator), and message history (memory)
2. **Agent loop** — an iterative cycle that calls the LLM, executes any requested tools, appends results to the conversation, and repeats until the LLM produces a final answer (`finish_reason == "stop"`)
3. **Action selection** — at each iteration, the LLM autonomously decides which tool to call (or none at all) based on the current task state and available tools
4. **Max iterations** — a safety limit that prevents infinite loops if the LLM never reaches a stopping point